<a href="https://colab.research.google.com/github/athirai-s/Maize-RL/blob/master/notebooks/po_ppo_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Apr 21 03:22:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/athirai-s/Maize-RL.git
%cd Maize-RL
!ls


Cloning into 'Maize-RL'...
remote: Enumerating objects: 433, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 433 (delta 41), reused 81 (delta 23), pack-reused 321 (from 2)
Receiving objects: 100% (433/433), 51.49 MiB | 39.29 MiB/s, done.
Resolving deltas: 100% (135/135), done.
/content/Maize-RL
14551_LMRL_Gym_Benchmarks_for_.pdf  LMRL-Gym		     scripts
double_t_maze_gray_walls.png	    maze_image_embedding.py


In [3]:
!ls LMRL-Gym/
!find . -name "JaxSeq" -type d 2>/dev/null
!grep -r "from JaxSeq" LMRL-Gym/llm_rl_scripts/maze/bc/ | head -3

environment.yml  llm_rl_scripts    setup_jaxseq.sh  tpu_vm_setup.sh
LICENSE		 README.md	   setup.py
LLM_RL		 requirements.txt  stockfish
LMRL-Gym/llm_rl_scripts/maze/bc/partially_observed_bc.py:from JaxSeq.bucket_manager import open_with_bucket as open
LMRL-Gym/llm_rl_scripts/maze/bc/partially_observed_bc.py:from JaxSeq.utils import convert_path, load_mesh, setup_experiment_save, MapIterable, BlockingStrategy, Padding, Truncation
LMRL-Gym/llm_rl_scripts/maze/bc/partially_observed_bc.py:from JaxSeq.utils import get_weight_decay_mask


In [4]:
%cd /content
!git clone https://github.com/Sea-Snell/JaxSeq.git
!ls JaxSeq/
%cd /content/Maize-RL

/content
Cloning into 'JaxSeq'...
remote: Enumerating objects: 572, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 572 (delta 9), reused 4 (delta 4), pack-reused 559 (from 1)
Receiving objects: 100% (572/572), 204.95 KiB | 3.06 MiB/s, done.
Resolving deltas: 100% (381/381), done.
environment.yml  JaxSeq   README.md	    setup.py
examples_jaxseq  LICENSE  requirements.txt  tpu_vm_setup.sh
/content/Maize-RL


In [5]:
!pip install --upgrade pip
!pip install jax[cuda11_pip]==0.4.17 jaxlib==0.4.17 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install numpy==1.24.4 scipy==1.9.0 chex==0.1.8 optax==0.1.7
!pip install nvidia-cudnn-cu11==8.6.0.163
!pip install transformers==4.26.1 tyro wandb flax tqdm jaxtyping dill cloudpickle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 MB 88.8 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.9/204.9 MB 107.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 44.5 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 119.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 118.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 61.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 MB 47.2 MB/s  0:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 109.3 MB/s  0

In [6]:
!apt-get install python3.9 python3.9-distutils python3.9-dev -y
!curl https://bootstrap.pypa.io/get-pip.py | python3.9
!python3.9 --version


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.9 libpython3.9-dev libpython3.9-minimal libpython3.9-stdlib
  python3.9-lib2to3 python3.9-minimal
Suggested packages:
  python3.9-venv binfmt-support
The following NEW packages will be installed:
  libpython3.9 libpython3.9-dev libpython3.9-minimal libpython3.9-stdlib
  python3.9 python3.9-dev python3.9-distutils python3.9-lib2to3
  python3.9-minimal
0 upgraded, 9 newly installed, 0 to remove and 42 not upgraded.
Need to get 12.2 MB of archives.
After this operation, 46.6 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.9-minimal amd64 3.9.25-1+jammy1 [838 kB]
Get:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 python3.9-minimal amd64 3.9.25-1+jammy1 [2,073 kB]
Get:3 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubunt

In [7]:
!python3.9 -m pip install --upgrade pip
!python3.9 -m pip install jax[cuda11_pip]==0.4.17 jaxlib==0.4.17 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!python3.9 -m pip install numpy==1.24.4 scipy==1.9.0 chex==0.1.8 optax==0.1.7
!python3.9 -m pip install nvidia-cudnn-cu11==8.6.0.163
!python3.9 -m pip install transformers==4.26.1 tyro wandb flax==0.7.4 tqdm jaxtyping dill cloudpickle

Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
  Using cached jax-0.4.17-py3-none-any.whl.metadata (23 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 MB 64.8 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.9/204.9 MB 78.3 MB/s  0:00:02
  Using cached nvidia_cublas_cu11-11.11.3.6-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu11-11.8.87-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cuda_nvcc_cu11-11.8.89-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu11-11.8.89-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cudnn_cu11-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cufft_cu11-10.9.0.58-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu11-11.4.1.48-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached 

In [9]:
%cd /content/JaxSeq
!python3.9 -m pip install -e . --no-deps
%cd /content/Maize-RL/LMRL-Gym
!python3.9 -m pip install -e . --no-deps
%cd /content/Maize-RL


/content/JaxSeq
Obtaining file:///content/JaxSeq
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for JaxSeq (pyproject.toml) ... done
  Created wheel for JaxSeq: filename=jaxseq-1.0.0-0.editable-py3-none-any.whl size=3924 sha256=05e4bed39844f74fb4d75ec56d37eb97ce73dd9dc4cf16298db1e1b7138aaf38
  Stored in directory: /tmp/pip-ephem-wheel-cache-kdmwtw4p/wheels/da/e2/96/39e864626a753124638ab4ea9c1d835f1d1f04d4a786407319
Successfully built JaxSeq
/content/Maize-RL/LMRL-Gym
Obtaining file:///content/Maize-RL/LMRL-Gym
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for LLM_RL (pyproject.toml) ... done
  Created wheel for LLM_RL: filename=l

In [12]:
%%writefile /tmp/check.py
import jax
print('JAX version:', jax.__version__)
print('JAX devices:', jax.devices())
import jax.numpy as jnp
x = jnp.ones((100, 100))
print('GPU compute works:', (x @ x).sum())
from JaxSeq.models.gpt2.load import load_params, ModelLoadMode
print('JaxSeq import OK')
from llm_rl_scripts.maze.bc.eval_bc import main
print('LMRL-Gym eval_bc import OK')


Writing /tmp/check.py


In [14]:
!python3.9 -m pip show jax jaxlib | grep -E "Name|Version|Location"
!python3.9 -m pip install gcsfs google-cloud-storage
!python3.9 -m pip install --force-reinstall --no-deps jax==0.4.17 jaxlib==0.4.17+cuda11.cudnn86 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html


Name: jax
Version: 0.4.30
Location: /usr/local/lib/python3.9/dist-packages
Name: jaxlib
Version: 0.4.30
Location: /usr/local/lib/python3.9/dist-packages
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 97.1 MB/s  0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 3.4.8
    Uninstalling cryptography-3.4.8:
      Successfully uninstalled cryptography-3.4.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26/26 [gcsfs]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxseq 1.0.0 requires Flask==2.2.1, which is not installed.
jaxseq 1.0.0 requires Flask-Cors==3.0.10, which is not installed.
jaxseq 1.0.0 requires frozendict==2.3.4, which is not installed.
jaxseq 1.0.0 requires h5py==3.7.0, which is not installed.
jaxseq 1.0.0 requires openai==0.27.2, wh

In [18]:
!python3.9 -m pip install --force-reinstall ipython
!python3.9 -c "import IPython; print(IPython.__file__)"


  Using cached decorator-5.2.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pygments-2.20.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.2/808.2 kB 25.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.5 MB/s  0:00:00
Using cached pygments-2.20.0-py3-none-any.whl (1.2 MB)
Using cached decorator-5.2.1-py3-none-any.whl (9.2 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: pygments
    Found existing installation: Pygments 2.20.0
    Uninstalling Pygments-2.20.0:
      Successfully uninstalled Pygments-2.20.0
  Attempting uninstall: decorator
    Found existing installation: decorator 5.2.1
    Uninstalling decora

In [20]:
!python3.9 -m pip install scikit-image
!python3.9 /tmp/check.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 144.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 161.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-image]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxseq 1.0.0 requires Flask==2.2.1, which is not installed.
jaxseq 1.0.0 requires Flask-Cors==3.0.10, which is not installed.
jaxseq 1.0.0 requires openai==0.27.2, which is not installed.
jaxseq 1.0.0 requires rouge-score==0.1.2, which is not installed.
jaxseq 1.0.0 requires sklearn==0.0, which is not installed.
jaxseq 1.0.0 requires streamlit==1.20.0, which is not installed.
jaxseq 1.0.0 requires termcolor==1.1.0, which is not installed.
jaxseq 1.0.0 requires torch==1.11, which is not installed.
llm-rl 1.0.0 requires chess, which

Download pretrained berkely checkpoint

In [21]:
!mkdir -p /content/checkpoints/ppo_po_pretrained
%cd /content/checkpoints/ppo_po_pretrained
!wget -q --show-progress https://rail.eecs.berkeley.edu/datasets/rl-llm-bench-dataset/maze/checkpoints/partially_observed/ppo/config.json
!wget -q --show-progress https://rail.eecs.berkeley.edu/datasets/rl-llm-bench-dataset/maze/checkpoints/partially_observed/ppo/params.msgpack
!wget -q --show-progress https://rail.eecs.berkeley.edu/datasets/rl-llm-bench-dataset/maze/checkpoints/partially_observed/ppo/train_state.msgpack
!ls -lh
%cd /content/Maize-RL

/content/checkpoints/ppo_po_pretrained
config.json         100%[===================>]     990  --.-KB/s    in 0s      
params.msgpack      100%[===================>] 259.74M  43.0MB/s    in 5.7s    
train_state.msgpack 100%[===================>]   1.01G  46.1MB/s    in 19s     
total 1.3G
-rw-r--r-- 1 root root  990 Dec 14  2023 config.json
-rw-r--r-- 1 root root 260M Dec 14  2023 params.msgpack
-rw-r--r-- 1 root root 1.1G Dec 14  2023 train_state.msgpack
/content/Maize-RL


In [22]:
!mkdir -p /content/outputs/eval_ppo_po_pretrained
%cd /content/Maize-RL/LMRL-Gym
!python3.9 -m llm_rl_scripts.maze.bc.eval_bc \
    PARAMS /content/checkpoints/ppo_po_pretrained \
    --outputs-path=/content/outputs/eval_ppo_po_pretrained/ \
    --no-fully-observed \
    --policy-n-rollouts=32 \
    --policy-bsize=1 \
    --policy-max-input-length=256 \
    --policy-max-output-length=8

/content/Maize-RL/LMRL-Gym
/usr/local/lib/python3.9/dist-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/usr/local/lib/python3.9/dist-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
{'model_load_mode': <ModelLoadMode.PARAMS: 'params'>, 'model_load_path': '/content/checkpoints/ppo_po_pretrained', 'data_mesh_shape': 1, 'fsdp_mesh_shape': 1, 'model_mesh_shape': -1, 'bf16_activations': Fa

In [25]:
!python3.9 -m llm_rl_scripts.maze.bc.eval_bc \
    PARAMS /content/checkpoints/ppo_po_pretrained \
    --outputs-path=/content/outputs/eval_ppo_po_long/ \
    --no-fully-observed \
    --policy-n-rollouts=32 \
    --policy-bsize=1 \
    --policy-max-input-length=1016 \
    --policy-max-output-length=8 \
    --maze-last-k=40 \
    --no-policy-do-sample


/usr/local/lib/python3.9/dist-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/usr/local/lib/python3.9/dist-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
{'model_load_mode': <ModelLoadMode.PARAMS: 'params'>, 'model_load_path': '/content/checkpoints/ppo_po_pretrained', 'data_mesh_shape': 1, 'fsdp_mesh_shape': 1, 'model_mesh_shape': -1, 'bf16_activations': False, 'maze_name': 'double_t